# Partie RDD + ETL (étapes 3 et 6)

Notebook de développement. Les cellules de préparation reprennent la SparkSession (étape 1), le chargement (étape 2) et le CA par magasin (étape 4), pour que ce notebook tourne seul.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum

spark = SparkSession.builder.appName("MiniProjet_Spark").master("local[2]").getOrCreate()
print("Master =", spark.sparkContext.master)

In [ ]:
# Préparation : DataFrame (étape 2) et CA par magasin (étape 4)
df = spark.read.csv("/home/jovyan/work/purchases.txt", sep="\t", header=False)
df = df.toDF("Date", "Heure", "Magasin", "Categorie", "Cout", "Paiement")
df = df.withColumn("Cout", col("Cout").cast("double"))

df_ca_magasin = df.groupBy("Magasin").agg(sum("Cout").alias("CA_total"))

# Étape 3 : Traitement avec les RDD

On repart du fichier brut `purchases.txt` avec l'API bas niveau de Spark (RDD). Chaque ligne contient 6 champs séparés par une tabulation : Date, Heure, Magasin, Catégorie, Coût, Paiement.

In [ ]:
sc = spark.sparkContext

# Création du RDD à partir du fichier texte
rdd = sc.textFile("/home/jovyan/work/purchases.txt")

# 1. Afficher les cinq premiers éléments
for ligne in rdd.take(5):
    print(ligne)

Chaque élément du RDD est pour l'instant une simple chaîne de caractères (une ligne du fichier).

In [ ]:
# 2. map() : découper chaque ligne en liste de champs
rdd_champs = rdd.map(lambda ligne: ligne.split("\t"))

# On écarte les éventuelles lignes mal formées (≠ 6 champs)
rdd_champs = rdd_champs.filter(lambda champs: len(champs) == 6)

rdd_champs.take(5)

In [ ]:
# 3. Extraire les couples (Magasin, Cout), avec Cout converti en nombre
rdd_magasin_cout = rdd_champs.map(lambda champs: (champs[2], float(champs[4])))

rdd_magasin_cout.take(5)

In [ ]:
# 4. filter() : garder les transactions dont le montant est > 100
rdd_sup_100 = rdd_magasin_cout.filter(lambda x: x[1] > 100)

print("Transactions > 100 :", rdd_sup_100.count())
rdd_sup_100.take(5)

In [ ]:
# 5. reduceByKey() : montant total des ventes par magasin
rdd_ca_magasin = rdd_magasin_cout.reduceByKey(lambda a, b: a + b)

print("Nombre de magasins :", rdd_ca_magasin.count())
# Affichage des 10 plus gros CA (tri décroissant)
for magasin, total in rdd_ca_magasin.sortBy(lambda x: x[1], ascending=False).take(10):
    print(f"{magasin:<20} {total:,.2f}")

In [ ]:
# Variante : total par magasin uniquement sur les ventes > 100
rdd_ca_sup_100 = rdd_sup_100.reduceByKey(lambda a, b: a + b)

rdd_ca_sup_100.sortBy(lambda x: x[1], ascending=False).take(5)

**Interprétation :** on obtient un chiffre d'affaires par magasin. Ces totaux sont les mêmes que ceux calculés avec la DataFrame API et Spark SQL (étapes 4 et 5), mais le code RDD est plus long : il faut découper les lignes et convertir les types à la main.

### Questions

**1. Différence entre `map()` et `filter()` ?**  
`map()` applique une fonction à chaque élément et renvoie un élément transformé pour chacun (même nombre d'éléments). `filter()` ne modifie pas les éléments : il garde seulement ceux qui vérifient une condition (moins d'éléments ou autant).

**2. Rôle de `reduceByKey()` ?**  
Sur un RDD de couples (clé, valeur), il regroupe les valeurs qui ont la même clé et les combine avec une fonction (ici l'addition). Il fait d'abord une réduction locale dans chaque partition, puis combine les résultats, ce qui limite les données échangées entre les partitions (shuffle).

**3. Différence entre une transformation et une action ?**  
Une transformation (`map`, `filter`, `reduceByKey`…) crée un nouveau RDD et est *paresseuse* (lazy) : Spark l'enregistre sans l'exécuter. Une action (`take`, `count`, `collect`…) déclenche réellement le calcul et renvoie un résultat au driver.

# Étape 6 : Mini-pipeline ETL avec PySpark

### Sauvegarde des résultats

In [ ]:
# 1. Récupérer la DataFrame du chiffre d'affaires par magasin (étape 4)
df_resultat = df_ca_magasin

df_resultat.orderBy("Magasin").show(10)

In [ ]:
# 2. Enregistrer au format Parquet (mode overwrite pour pouvoir réexécuter)
output_path = "/home/jovyan/work/output/vente_par_magasin"

df_resultat.write.mode("overwrite").parquet(output_path)

In [ ]:
# 3. Vérifier que les fichiers ont bien été créés
import os
print(os.listdir(output_path))

Le dossier contient un ou plusieurs fichiers `part-*.parquet` (les données) et un fichier `_SUCCESS` qui indique que l'écriture s'est bien terminée.

### Vérification des données enregistrées

In [ ]:
# 1. Relire les données Parquet dans une nouvelle DataFrame
df_verification = spark.read.parquet(output_path)

# 2. Afficher les premières lignes
df_verification.orderBy("Magasin").show(10)

In [ ]:
# 3. Vérifier le schéma
df_verification.printSchema()

In [ ]:
# 4. Comparer avec les résultats d'avant l'enregistrement
print("Lignes avant :", df_resultat.count())
print("Lignes après :", df_verification.count())
print("Même schéma  :", df_resultat.schema == df_verification.schema)

# Différences dans les deux sens (0 = données identiques)
diff_1 = df_resultat.exceptAll(df_verification).count()
diff_2 = df_verification.exceptAll(df_resultat).count()
print("Lignes différentes :", diff_1 + diff_2)

**Interprétation :** même nombre de lignes, même schéma (Magasin : string, CA_total : double) et aucune ligne différente. Le format Parquet a conservé les données et leurs types. L'ordre d'affichage peut varier, c'est pourquoi on trie par magasin avant de comparer visuellement.

### Questions

**1. Phases d'un pipeline ETL, et où on les trouve dans le projet :**
- **Extract** (extraire) : lecture de `purchases.txt` (étape 2 en DataFrame, étape 3 en RDD).
- **Transform** (transformer) : nommage des colonnes, conversion de `Cout` en double, filtres, agrégations par magasin et par catégorie (étapes 3, 4 et 5).
- **Load** (charger) : écriture du CA par magasin au format Parquet (étape 6).

**2. Pourquoi utiliser Parquet en Big Data ?**  
C'est un format colonnaire, compressé, qui garde le schéma. Spark ne lit que les colonnes utiles, les fichiers sont plus petits et les lectures plus rapides. Il se découpe aussi bien pour un traitement en parallèle.

**3. Différence entre CSV et Parquet ?**  
CSV : texte lisible, stocké ligne par ligne, sans types (tout est lu comme du texte, il faut ré-inférer ou caster), sans compression. Parquet : binaire, stocké par colonnes, compressé, avec le schéma et les types inclus.

**4. Intérêt du mode `overwrite` ?**  
Il remplace le contenu du dossier s'il existe déjà. On peut donc relancer le notebook sans erreur (le mode par défaut échoue si le dossier existe) et sans accumuler d'anciens résultats.

**5. Différence entre `show()` et l'enregistrement en Parquet ?**  
`show()` affiche seulement quelques lignes à l'écran, et rien n'est conservé après l'arrêt de la session. Enregistrer en Parquet écrit toutes les données sur disque : elles sont persistantes et réutilisables par d'autres traitements.